# 🔍 Week 6: Named Entity Recognition (NER)

## Tujuan Pembelajaran:
1. Memahami konsep **Named Entity Recognition (NER)** — mengidentifikasi dan mengklasifikasikan entitas bernama (nama orang, lokasi, organisasi, dll.) dalam teks
2. Menggunakan model **spaCy pretrained** (`en_core_web_sm`) untuk NER pada teks Inggris
3. Mengenal jenis-jenis entitas dalam skema **OntoNotes** yang digunakan spaCy
4. Menerapkan **EntityRuler** untuk menambahkan aturan NER berbasis pola (phrase & token matching)
5. Melakukan NER pada **teks bahasa Indonesia** menggunakan model **BERT** (`cahya/bert-base-indonesian-NER`)
6. Menerapkan NER ke seluruh dataset ulasan Honest Review

---
> 📂 **Dataset**: Honest Review (`cleandata.csv`) — kolom `text_final` (teks yang sudah dipreproses)
> File ini berada di folder `Week 3/` dalam workspace yang sama.

## 1) Install Library

Install spaCy, model bahasa Inggris, dan library Transformers untuk model NER bahasa Indonesia.

In [ ]:
!pip install spacy transformers sentencepiece pandas

In [ ]:
# Download model spaCy bahasa Inggris
!python -m spacy download en_core_web_sm

## 2) NER dengan spaCy — Model Pretrained

spaCy menyediakan model NER yang sudah dilatih pada corpus besar. Model `en_core_web_sm` dapat mengenali entitas seperti nama orang, organisasi, lokasi, tanggal, dan lainnya dari teks Inggris.

Pipeline spaCy terdiri dari:
1. `tok2vec` — representasi token
2. `tagger` — POS tagging
3. `parser` — dependency parsing
4. `ner` — named entity recognition

In [ ]:
import spacy
from spacy import displacy

# Load model spaCy bahasa Inggris
nlp = spacy.load('en_core_web_sm')
print("Model spaCy berhasil dimuat.")
print(f"Pipeline: {nlp.pipe_names}")

In [ ]:
# Contoh NER pada teks Inggris
text = (
    "George Washington was the first president of the United States. "
    "He was born on February 22, 1732, in Virginia. "
    "He led the Continental Army and later served two terms as president."
)

doc = nlp(text)

print("Entitas yang ditemukan:")
print(f"{'Entitas':<30} {'Label':<12} {'Keterangan'}")
print('-' * 65)
for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<12} {spacy.explain(ent.label_)}")

In [ ]:
# Visualisasi NER dengan displacy
displacy.render(doc, style='ent', jupyter=True)

## 3) Jenis Entitas OntoNotes (spaCy)

spaCy menggunakan skema entity **OntoNotes 5**. Berikut daftar lengkap entitas yang didukung:

| Label      | Keterangan |
|------------|------------|
| `PERSON`   | Nama orang, termasuk tokoh fiksi |
| `NORP`     | Kebangsaan, kelompok agama atau politik |
| `FAC`      | Fasilitas: bangunan, bandara, jembatan, jalan |
| `ORG`      | Perusahaan, lembaga, instansi pemerintah |
| `GPE`      | Negara, kota, provinsi |
| `LOC`      | Lokasi non-GPE: gunung, lautan, sungai |
| `PRODUCT`  | Objek, kendaraan, makanan |
| `EVENT`    | Perang, pertempuran, olahraga, dll. |
| `WORK_OF_ART` | Judul buku, lagu, karya seni |
| `LAW`      | Peraturan, undang-undang |
| `LANGUAGE` | Bahasa yang disebutkan |
| `DATE`     | Tanggal atau periode waktu |
| `TIME`     | Waktu lebih kecil dari sehari |
| `PERCENT`  | Persentase |
| `MONEY`    | Nilai moneter |
| `QUANTITY` | Ukuran, berat, jarak |
| `ORDINAL`  | "pertama", "kedua", dll. |
| `CARDINAL` | Angka yang tidak termasuk kategori lain |

In [ ]:
# Cetak semua entitas yang didukung model
print("Entitas yang dikenali oleh model en_core_web_sm:")
for label in nlp.get_pipe('ner').labels:
    print(f"  {label:<15} → {spacy.explain(label)}")

## 4) EntityRuler — Phrase Matching

**EntityRuler** memungkinkan kita mendefinisikan aturan NER berbasis pola, bukan statistik. Berguna ketika ada daftar nama yang sudah diketahui (misalnya: nama presiden, nama kota).

**Phrase patterns** cocok untuk pencocokan string eksak.

In [ ]:
# Tambahkan EntityRuler ke pipeline — SEBELUM komponen 'ner'
ruler = nlp.add_pipe("entity_ruler", before="ner", name="presidents")

presidents = [
    "George Washington", "John Adams", "Thomas Jefferson", "Abraham Lincoln",
    "Franklin D. Roosevelt", "Harry S. Truman", "Dwight D. Eisenhower",
    "John F. Kennedy", "Lyndon B. Johnson", "Richard Nixon",
    "Jimmy Carter", "Ronald Reagan", "George H.W. Bush",
    "Bill Clinton", "George W. Bush", "Barack Obama", "Donald Trump"
]

def make_patterns(name_list, label):
    """Ubah list nama menjadi format pola EntityRuler."""
    return [{"label": label, "pattern": name} for name in name_list]

ruler.add_patterns(make_patterns(presidents, "PRES"))
print(f"EntityRuler ditambahkan. Pipeline: {nlp.pipe_names}")
print(f"Jumlah pola terdaftar: {len(ruler.patterns)}")

In [ ]:
# Test EntityRuler
text2 = (
    "Barack Obama and George W. Bush both attended the inauguration of Donald Trump "
    "at the Capitol in Washington D.C. in January 2017."
)
doc2 = nlp(text2)

print("Entitas yang ditemukan (dengan EntityRuler):")
print(f"{'Entitas':<30} {'Label'}")
print('-' * 45)
for ent in doc2.ents:
    print(f"{ent.text:<30} {ent.label_}")
print()
displacy.render(doc2, style='ent', jupyter=True)

## 5) EntityRuler — Token Patterns (Aturan Berbasis Atribut Token)

**Token patterns** lebih fleksibel — bisa menggunakan regex dan atribut seperti `LOWER`, `POS`, `SHAPE`. Cocok untuk mencocokkan berbagai variasi penulisan nama yang sama.

In [ ]:
# Hapus EntityRuler lama dan buat ulang dengan token patterns
nlp.remove_pipe('presidents')
ruler2 = nlp.add_pipe("entity_ruler", before="ner", name="presidents")

token_patterns = [
    # 2 token: "george bush"
    {"id": "PRES41", "label": "PRES",
     "pattern": [{"LOWER": "george"}, {"LOWER": "bush"}]},
    # 3 token: "george hw bush", "george h.w. bush"
    {"id": "PRES41", "label": "PRES",
     "pattern": [{"LOWER": "george"}, {"LOWER": {"REGEX": "h(\\.w\\.?|w)"}}, {"LOWER": "bush"}]},
    # 4 token: "george herbert walker bush"
    {"id": "PRES41", "label": "PRES",
     "pattern": [{"LOWER": "george"}, {"LOWER": {"REGEX": "h(erbert|\\.?)"}},
                 {"LOWER": {"REGEX": "w(alker|\\.?)"}}, {"LOWER": "bush"}]},
]

ruler2.add_patterns(token_patterns)

# Uji berbagai format nama
text3 = (
    "Known names: George Bush, George HW Bush, George H.W. Bush, "
    "George Herbert Walker Bush."
)
doc3 = nlp(text3)

print("Entitas yang ditemukan (Token Patterns):")
for ent in doc3.ents:
    print(f"  {ent.text!r:<40} → {ent.label_}")

displacy.render(doc3, style='ent', jupyter=True)

## 6) NER Bahasa Indonesia dengan BERT

Untuk teks bahasa Indonesia, kita menggunakan model **`cahya/bert-base-indonesian-NER`** — model BERT yang dilatih khusus untuk NER pada teks Indonesia. Model ini mengenali entitas:

| Label | Keterangan |
|-------|------------|
| `PER` | Nama orang (Person) |
| `ORG` | Organisasi |
| `LOC` | Lokasi |
| `QTY` | Kuantitas |
| `EVT` | Event/Kejadian |
| `PRD` | Produk |
| `TIM` | Waktu |

In [ ]:
from transformers import pipeline, AutoTokenizer

print("Loading model NER bahasa Indonesia...")
tokenizer = AutoTokenizer.from_pretrained("cahya/bert-base-indonesian-NER")
ner_pipeline_id = pipeline(
    "ner",
    model="cahya/bert-base-indonesian-NER",
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)
print("Model berhasil dimuat!")

In [ ]:
# Uji NER pada kalimat bahasa Indonesia
teks_id = "Jokowi mengunjungi Bandung untuk meresmikan proyek infrastruktur baru bersama Menteri PUPR Basuki."

results = ner_pipeline_id(teks_id)
print("=== Hasil NER Bahasa Indonesia ===")
print(f"Teks: {teks_id}\n")
print(f"{'Token':<25} {'Entitas':<10} {'Score'}")
print('-' * 50)
for r in results:
    print(f"{r['word']:<25} {r['entity_group']:<10} {r['score']:.4f}")

## 7) Terapkan NER ke Seluruh Dataset Honest Review

Load dataset `cleandata.csv` dan terapkan model NER bahasa Indonesia ke kolom `text_final`. Karena teks sudah dipreproses (stemmed), kita langsung terapkan tanpa cleaning tambahan.

In [ ]:
import pandas as pd

file_path = '../Week 3/cleandata.csv'
df_text = pd.read_csv(file_path)
print(f"Jumlah data: {len(df_text)}")
df_text[['content', 'text_final', 'score']].head()

In [ ]:
def perform_ner(text, max_length=256):
    """
    Terapkan NER pada satu teks.
    max_length dibatasi untuk efisiensi karena teks sudah di-preproses (pendek).
    Return string entitas yang ditemukan (format: 'kata (LABEL)').
    """
    cleaned = str(text)[:max_length]
    results = ner_pipeline_id(cleaned, aggregation_strategy="simple")
    entities = list(set(f"{r['word']} ({r['entity_group']})" for r in results))
    return ', '.join(entities) if entities else '-'

def safe_perform_ner(text):
    """Wrapper dengan penanganan error."""
    try:
        return perform_ner(text)
    except Exception as e:
        return f"Error: {str(e)[:50]}"

print("Fungsi NER siap digunakan.")

In [ ]:
# Terapkan NER ke seluruh dataset
# Catatan: Proses ini dapat memakan beberapa menit
print("Memproses NER untuk seluruh dataset...")
df_text['NER_Results'] = df_text['text_final'].apply(safe_perform_ner)
print("NER selesai!")
df_text[['text_final', 'NER_Results']].head(5)

## 8) Simpan Hasil NER

Simpan DataFrame dengan hasil NER ke file CSV.

In [ ]:
output_file = 'ner_results_honest.csv'
df_text.to_csv(output_file, index=False)
print(f"Hasil NER disimpan ke '{output_file}'")
print(f"Total ulasan diproses: {len(df_text):,}")
df_text